# Fisher-KPP Disk Dataset Exploration

This notebook loads the committed `train.npz` and `test.npz` files for the Fisher-KPP disk case, reports the snapshot counts, visualizes the first few saved training snapshots, and documents the parameter values used for every training trajectory and the held-out testing trajectory.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'train.npz').exists():
    ROOT = Path('data/fisher_kpp_disk').resolve()
TRAIN_PATH = ROOT / 'train.npz'
TEST_PATH = ROOT / 'test.npz'

train = np.load(TRAIN_PATH, allow_pickle=False)
test = np.load(TEST_PATH, allow_pickle=False)
train_meta = json.loads(str(train['metadata_json']))
test_meta = json.loads(str(test['metadata_json']))

PARAMETER_DESCRIPTIONS = {
    'diffusion': 'Diffusion coefficient D in the reaction-diffusion model.',
    'growth_rate': 'Logistic growth coefficient r.',
    'initial_amplitude': 'Amplitude scaling for the smooth blob-based initial condition.',
    'initial_condition_seed': 'Deterministic seed used to place and scale the initial Gaussian blobs.'
}

def split_summary(dataset, metadata):
    print(f"Split: {dataset['split_name'].item()}")
    print(f"  trajectories: {dataset['parameter_matrix'].shape[0]}")
    print(f"  total snapshots: {dataset['states'].shape[0]}")
    print(f"  snapshots per trajectory: {dataset['trajectory_lengths'].tolist()[:5]}{' ...' if len(dataset['trajectory_lengths']) > 5 else ''}")
    print(f"  state shape per snapshot: {dataset['states'].shape[1:]}")
    print(f"  saved time window: [{dataset['times'].min():.4f}, {dataset['times'].max():.4f}]")
    print(f"  steady-time estimate range: [{dataset['steady_time_estimates'].min():.4f}, {dataset['steady_time_estimates'].max():.4f}]")
    print(f"  equation: {metadata['equation']}")

def trajectory_block(dataset, trajectory_index):
    start = int(dataset['trajectory_offsets'][trajectory_index])
    stop = int(dataset['trajectory_offsets'][trajectory_index + 1])
    return dataset['states'][start:stop], dataset['times'][start:stop]

split_summary(train, train_meta)
print()
split_summary(test, test_meta)


## Parameter Documentation

The parameter rows below document the diffusion, growth, and initialization values attached to each transient rollout, together with the exact saved time span covered by that rollout.

In [ ]:
parameter_names = train['parameter_names'].tolist()
print('Parameter names and meanings:')
for name in parameter_names:
    print(f"- {name}: {PARAMETER_DESCRIPTIONS.get(name, 'No description available.')}")

print('\nTraining trajectory parameter table:')
header = ['traj_id', 't_start', 't_end', *parameter_names]
print(' | '.join(header))
for traj_id, params in enumerate(train['parameter_matrix']):
    states_i, times_i = trajectory_block(train, traj_id)
    row = [traj_id, f"{times_i[0]:.4f}", f"{times_i[-1]:.4f}", *[f"{value:.6g}" for value in params]]
    print(' | '.join(map(str, row)))

print('\nHeld-out testing trajectory parameters:')
test_states_0, test_times_0 = trajectory_block(test, 0)
print('time window:', f"[{test_times_0[0]:.4f}, {test_times_0[-1]:.4f}]")
for name, value in zip(parameter_names, test['parameter_matrix'][0]):
    print(f"- {name}: {value:.6g}")


## Exploratory Plots

The first figure shows selected early snapshots from the first training trajectory. The second figure reduces every flattened training snapshot to simple summary statistics so trends can be inspected quickly.

In [ ]:
x = train['x']
y = train['y']
mask = train['mask'].astype(bool)
states0, times0 = trajectory_block(train, 0)
plot_indices = np.linspace(0, len(times0) - 1, min(4, len(times0)), dtype=int)

fig, axes = plt.subplots(1, len(plot_indices), figsize=(4 * len(plot_indices), 4), constrained_layout=True)
if len(plot_indices) == 1:
    axes = [axes]
for ax, idx in zip(axes, plot_indices):
    image = np.where(mask, states0[idx], np.nan)
    im = ax.imshow(image, origin='lower', extent=[x.min(), x.max(), y.min(), y.max()], cmap='viridis', vmin=0.0)
    ax.set_title(f"t = {times0[idx]:.4f}")
    ax.set_xlabel('x')
    ax.set_ylabel('y')
fig.colorbar(im, ax=axes, shrink=0.85)
plt.show()

flat_train = train['states'].reshape(train['states'].shape[0], -1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(train['times'], flat_train.mean(axis=1))
axes[0].set_title('Mean over all training snapshots')
axes[0].set_xlabel('time')
axes[0].grid(True, alpha=0.25)
axes[1].plot(train['times'], flat_train.min(axis=1))
axes[1].set_title('Minimum over all training snapshots')
axes[1].set_xlabel('time')
axes[1].grid(True, alpha=0.25)
axes[2].plot(train['times'], flat_train.max(axis=1))
axes[2].set_title('Maximum over all training snapshots')
axes[2].set_xlabel('time')
axes[2].grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(train['parameter_matrix'][:, 1], bins=12, alpha=0.8, edgecolor='black')
plt.title('Training distribution of growth-rate values')
plt.xlabel('growth_rate')
plt.ylabel('count')
plt.grid(True, alpha=0.25)
plt.show()
